In [34]:
import os
import time
import json
import requests
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sina.config.paths import GAS_DATA
from sina.config.credentials import gasolina_api_rest, google_api_key

CACHE_FILE = Path("gasolineras_coordenadas.json")
base_dir = GAS_DATA
CIUDAD  = "hermosillo"
ESTADO  = "sonora"
PAIS    = "mexico"

In [35]:
params = {
    "entidadId"  : "26",
    "municipioId": "030",
}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer"   : "https://www.cne.gob.mx/ConsultaPrecios/GasolinasyDiesel/GasolinasyDiesel.html"
}

response = requests.get(gasolina_api_rest, params=params, headers=headers)

print(f"Status  : {response.status_code}")
print(f"Size    : {len(response.content) / 1024:.1f} KB")  
print(f"Content : {response.headers.get('Content-Type')}")

data = response.json()
print(f"\nTipo    : {type(data)}")

if isinstance(data, list):
    print(f"Registros : {len(data)}")
    print(f"\nPrimer registro:")
    print(json.dumps(data[0], indent=2, ensure_ascii=False))

elif isinstance(data, dict):
    print(f"Keys : {list(data.keys())}")

Status  : 200
Size    : 111.7 KB
Content : application/json; charset=utf-8

Tipo    : <class 'dict'>
Keys : ['Success', 'Errors', 'Value']


In [36]:
print(f"Success : {data['Success']}")
print(f"Errors  : {data['Errors']}")
print(f"Registros en Value: {len(data['Value'])}")
print(f"\nPrimer registro:")
print(json.dumps(data['Value'][0], indent=2, ensure_ascii=False))

Success : True
Errors  : None
Registros en Value: 391

Primer registro:
{
  "Numero": "PL/3438/EXP/ES/2015",
  "Direccion": "Ángel García Aburto Y Dr. Domingo Olivares",
  "Producto": "Gasolinas",
  "SubProducto": "Premium (con un índice de octano ([RON+MON]/2) mínimo de 91)",
  "PrecioVigente": 29.99,
  "EntidadFederativaId": 26,
  "MunicipioId": 30,
  "Nombre": "SERVICIOS Z3 S.A. DE C.V."
}


In [37]:
df_cne = pd.DataFrame(data["Value"])
print(df_cne.shape)
print(df_cne.columns.tolist())
df_cne.head()

(391, 8)
['Numero', 'Direccion', 'Producto', 'SubProducto', 'PrecioVigente', 'EntidadFederativaId', 'MunicipioId', 'Nombre']


,Numero,Direccion,Producto,SubProducto,PrecioVigente,EntidadFederativaId,MunicipioId,Nombre
0,PL/3438/EXP/ES/2015,Ángel García Aburto Y Dr. Domingo Olivares,Gasolinas,Premium (con un índice de octano ([RON+MON]/2)...,29.99,26,30,SERVICIOS Z3 S.A. DE C.V.
1,PL/3438/EXP/ES/2015,Ángel García Aburto Y Dr. Domingo Olivares,Gasolinas,Regular (con un índice de octano ([RON+MON]/2)...,23.99,26,30,SERVICIOS Z3 S.A. DE C.V.
2,PL/20952/EXP/ES/2018,AVE MAR DE CORTES NÚMERO 8910,Diésel,Diésel Automotriz [contenido mayor de azufre a...,29.90,26,30,ESTACION DE SERVICIO ERSAL SA DE CV
3,PL/20952/EXP/ES/2018,AVE MAR DE CORTES NÚMERO 8910,Gasolinas,Premium (con un índice de octano ([RON+MON]/2)...,29.10,26,30,ESTACION DE SERVICIO ERSAL SA DE CV
4,PL/20952/EXP/ES/2018,AVE MAR DE CORTES NÚMERO 8910,Gasolinas,Regular (con un índice de octano ([RON+MON]/2)...,23.99,26,30,ESTACION DE SERVICIO ERSAL SA DE CV


In [38]:
print(df_cne["SubProducto"].unique())

['Premium (con un índice de octano ([RON+MON]/2) mínimo de 91)'
 'Regular (con un índice de octano ([RON+MON]/2) mínimo de 87)'
 'Diésel Automotriz [contenido mayor de azufre a 15 mg/kg y contenido máximo de azufre de 500 mg/kg]'
 'Diésel' 'Premium (con contenido mínimo de 92 octanos)'
 'Regular (con contenido menor a 92 octanos)'
 'Diésel de Ultra Bajo Azufre (DUBA) [contenido máximo de azufre de 15 mg/kg]']


In [39]:
mapa_productos = {
    sp: ("Premium" if "Premium" in sp 
         else "Magna" if "Regular" in sp 
         else "Diesel" if "Diésel" in sp  
         else "Otro")
    for sp in df_cne["SubProducto"].unique()
}

print("Mapeo encontrado:")
for k, v in mapa_productos.items():
    print(f"  {v:10} ← {k}")

df_cne["Combustible"] = df_cne["SubProducto"].map(mapa_productos)

Mapeo encontrado:
  Premium    ← Premium (con un índice de octano ([RON+MON]/2) mínimo de 91)
  Magna      ← Regular (con un índice de octano ([RON+MON]/2) mínimo de 87)
  Diesel     ← Diésel Automotriz [contenido mayor de azufre a 15 mg/kg y contenido máximo de azufre de 500 mg/kg]
  Diesel     ← Diésel
  Premium    ← Premium (con contenido mínimo de 92 octanos)
  Magna      ← Regular (con contenido menor a 92 octanos)
  Diesel     ← Diésel de Ultra Bajo Azufre (DUBA) [contenido máximo de azufre de 15 mg/kg]


In [40]:
df_pivot = df_cne.pivot_table(
    index   = ["Numero", "Nombre", "Direccion"],
    columns = "Combustible",
    values  = "PrecioVigente",
    aggfunc = "first"
).reset_index()

df_pivot.columns.name = None

for col in ["Magna", "Premium", "Diesel"]:
    if col not in df_pivot.columns:
        df_pivot[col] = None

print(f"Gasolineras únicas : {len(df_pivot)}")
df_pivot.head(10)

Gasolineras únicas : 157


,Numero,Nombre,Direccion,Diesel,Magna,Premium
0,PL/10156/EXP/ES/2015,FEDERICO LÓPEZ LÓPEZ,Carretera Ures-Hermosillo Km 3.5,28.99,NaN,28.99
1,PL/10244/EXP/ES/2015,GP GASOLINERA PERINORTE S.A. DE C.V.,Avenida Camino del Seri 1242,NaN,NaN,29.00
2,PL/10321/EXP/ES/2015,SERVICIOS DEL REAL DEL ALAMITO S.A. DE C.V.,Carretera Hermosillo Ures Entronq.Carretera de...,28.99,23.89,28.79
3,PL/10327/EXP/ES/2015,SERVICIO EL KENO S.A. DE C.V.,José Ma Mendoza 794,28.59,22.93,27.99
4,PL/10366/EXP/ES/2015,"ESTACION PIRU, S.A. DE C.V.",Xolotl No. 244,NaN,23.99,27.99
5,PL/10370/EXP/ES/2015,MIGRAG GRUPO CONSTUCTOR S.A. DE C.V.,Periférico Norte 4,NaN,22.99,27.90
6,PL/10696/EXP/ES/2015,GASOLINERA RS S.A. DE C.V.,Margarita Maza de Juárez Y José López Portillo Sn,NaN,23.49,27.99
7,PL/10740/EXP/ES/2015,ESTACIÓN DE SERVICIO QUIROGA S.A. DE C.V.,Boulevard Luis Donaldo Colosio 862 Esquinaboul...,NaN,22.99,28.99
8,PL/10761/EXP/ES/2015,"CORPORATIVO ENERVISION, S.A.P.I. DE C.V.",Boulevard López Portillo No. 189,29.99,24.29,29.99
9,PL/10763/EXP/ES/2015,"CORPORATIVO ENERVISION, S.A.P.I. DE C.V.",Boulevard Gomez Farias Y Boulevard Ignacio Sot...,NaN,24.29,29.99


In [41]:
df_pivot["Direccion_completa"] = (
    df_pivot["Direccion"]+ ", " + CIUDAD + ", " + ESTADO + ", " + PAIS
)

df_pivot.head()

,Numero,Nombre,Direccion,Diesel,Magna,Premium,Direccion_completa
0,PL/10156/EXP/ES/2015,FEDERICO LÓPEZ LÓPEZ,Carretera Ures-Hermosillo Km 3.5,28.99,NaN,28.99,"Carretera Ures-Hermosillo Km 3.5, hermosillo, ..."
1,PL/10244/EXP/ES/2015,GP GASOLINERA PERINORTE S.A. DE C.V.,Avenida Camino del Seri 1242,NaN,NaN,29.00,"Avenida Camino del Seri 1242, hermosillo, sono..."
2,PL/10321/EXP/ES/2015,SERVICIOS DEL REAL DEL ALAMITO S.A. DE C.V.,Carretera Hermosillo Ures Entronq.Carretera de...,28.99,23.89,28.79,Carretera Hermosillo Ures Entronq.Carretera de...
3,PL/10327/EXP/ES/2015,SERVICIO EL KENO S.A. DE C.V.,José Ma Mendoza 794,28.59,22.93,27.99,"José Ma Mendoza 794, hermosillo, sonora, mexico"
4,PL/10366/EXP/ES/2015,"ESTACION PIRU, S.A. DE C.V.",Xolotl No. 244,NaN,23.99,27.99,"Xolotl No. 244, hermosillo, sonora, mexico"


In [42]:
df_pivot[df_pivot['Nombre'] == "FONDO EMPRESARIAL S4M, S.A. DE C.V."]

,Numero,Nombre,Direccion,Diesel,Magna,Premium,Direccion_completa
36,PL/24968/EXP/ES/2023,"FONDO EMPRESARIAL S4M, S.A. DE C.V.",Blvd. Solidaridad No. 1120,28.99,23.99,28.99,"Blvd. Solidaridad No. 1120, hermosillo, sonora..."
43,PL/25364/EXP/ES/2023,"FONDO EMPRESARIAL S4M, S.A. DE C.V.","Carretera a la Colorada No. 189, esquina con B...",28.99,23.99,28.99,"Carretera a la Colorada No. 189, esquina con B..."


In [43]:
def get_google_coordinates(address: str, api_key: str):
    """
    Calls the Google Maps Geocoding API to get latitude and longitude.
    """
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": api_key
    }
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if data.get('status') == 'OK':
            location = data['results'][0]['geometry']['location']
            return location['lat'], location['lng']
        else:
            print(f"⚠️ API returned {data.get('status')} for address: {address}")
            return None, None
            
    except requests.exceptions.RequestException as e:
        print(f"❌ Network error while geocoding: {e}")
        return None, None

def geocode_dataframe(df: pd.DataFrame, api_key: str, cache_path: Path) -> pd.DataFrame:
    """
    Reads a cache file. If the gas station is not in cache, it calls Google API.
    Updates the cache file and merges coordinates back into the DataFrame.
    """
    if cache_path.exists():
        print(f"📂 Loading existing cache from {cache_path}")
        with open(cache_path, "r", encoding="utf-8") as f:
            coords_cache = json.load(f)
    else:
        print("✨ No cache found. Creating a new one...")
        coords_cache = {}

    new_entries = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Geocodificando"):
        key     = row["Numero"]
        address = row["Direccion_completa"]

        if key not in coords_cache:
            lat, lon = get_google_coordinates(address, api_key)
            coords_cache[key] = {
                "Nombre"             : row["Nombre"],
                "Direccion"          : row["Direccion"],
                "Direccion_completa" : address,
                "Latitud"            : lat,
                "Longitud"           : lon,
            }
            new_entries += 1
            time.sleep(0.05)

    if new_entries > 0:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(coords_cache, f, indent=2, ensure_ascii=False)
        print(f"💾 Cache actualizado: {new_entries} nuevas entradas")
    else:
        print("✅ Todo ya estaba en cache, no se hicieron llamadas a la API")

    coords_df = pd.DataFrame.from_dict(coords_cache, orient="index")
    coords_df.index.name = "Numero"
    coords_df = coords_df.reset_index()[["Numero", "Latitud", "Longitud"]]

    df_final = df.merge(coords_df, on="Numero", how="left")
    return df_final

In [45]:
output_dir = GAS_DATA / CIUDAD
CACHE_FILE = output_dir / Path(f"gasolineras_{CIUDAD}.json")

df_geocoded = geocode_dataframe(df_pivot, google_api_key, CACHE_FILE)

fallidas = df_geocoded[df_geocoded["Latitud"].isnull()]
if not fallidas.empty:
    print(f"\n Advertencia: No se encontraron coordenadas para {len(fallidas)} gasolineras.")

df_geocoded.head()

✨ No cache found. Creating a new one...


Geocodificando: 100%|██████████| 157/157 [01:12<00:00,  2.16it/s]

💾 Cache actualizado: 157 nuevas entradas


,Numero,Nombre,Direccion,Diesel,Magna,Premium,Direccion_completa,Latitud,Longitud
0,PL/10156/EXP/ES/2015,FEDERICO LÓPEZ LÓPEZ,Carretera Ures-Hermosillo Km 3.5,28.99,NaN,28.99,"Carretera Ures-Hermosillo Km 3.5, hermosillo, ...",29.082093,-110.942156
1,PL/10244/EXP/ES/2015,GP GASOLINERA PERINORTE S.A. DE C.V.,Avenida Camino del Seri 1242,NaN,NaN,29.00,"Avenida Camino del Seri 1242, hermosillo, sono...",29.056668,-111.015853
2,PL/10321/EXP/ES/2015,SERVICIOS DEL REAL DEL ALAMITO S.A. DE C.V.,Carretera Hermosillo Ures Entronq.Carretera de...,28.99,23.89,28.79,Carretera Hermosillo Ures Entronq.Carretera de...,29.145423,-111.023923
3,PL/10327/EXP/ES/2015,SERVICIO EL KENO S.A. DE C.V.,José Ma Mendoza 794,28.59,22.93,27.99,"José Ma Mendoza 794, hermosillo, sonora, mexico",29.101898,-110.993253
4,PL/10366/EXP/ES/2015,"ESTACION PIRU, S.A. DE C.V.",Xolotl No. 244,NaN,23.99,27.99,"Xolotl No. 244, hermosillo, sonora, mexico",29.024009,-110.946251


In [46]:
import folium
import pandas as pd
import numpy as np
from branca.colormap import LinearColormap

# ============================================================
# 1. PREPARAR DATOS
# ============================================================

# Asegurarnos que los precios son numéricos
for col in ["Magna", "Premium", "Diesel"]:
    df_geocoded[col] = pd.to_numeric(df_geocoded[col], errors="coerce")

# ============================================================
# 2. FUNCIÓN PRINCIPAL DEL MAPA
# ============================================================

def crear_mapa_gasolineras(df: pd.DataFrame, combustible: str = "Magna") -> folium.Map:
    """
    Crea un mapa interactivo de gasolineras en Hermosillo
    coloreadas por precio (verde=barato, rojo=caro).
    
    Args:
        df          : DataFrame con columnas Latitud, Longitud y precios
        combustible : "Magna", "Premium" o "Diesel"
    """
    
    # Filtrar solo gasolineras con datos válidos para ese combustible
    df_filtrado = df.dropna(subset=["Latitud", "Longitud", combustible]).copy()
    
    precio_min = df_filtrado[combustible].min()
    precio_max = df_filtrado[combustible].max()
    precio_avg = df_filtrado[combustible].mean()
    
    print(f"⛽ Combustible : {combustible}")
    print(f"🟢 Más barato  : ${precio_min:.2f}")
    print(f"🔴  Más caro    : ${precio_max:.2f}")
    print(f"📊 Promedio    : ${precio_avg:.2f}")
    print(f"📍 Gasolineras : {len(df_filtrado)}")
    
    # ----------------------------------------------------------
    # Colormap: verde (barato) → amarillo → rojo (caro)
    # ----------------------------------------------------------
    colormap = LinearColormap(
        colors=["#2ecc71", "#f1c40f", "#e74c3c"],
        vmin=precio_min,
        vmax=precio_max,
        caption=f"Precio {combustible} (MXN)"
    )
    
    # ----------------------------------------------------------
    # Centro del mapa: promedio de coordenadas
    # ----------------------------------------------------------
    centro_lat = df_filtrado["Latitud"].mean()
    centro_lon = df_filtrado["Longitud"].mean()
    
    mapa = folium.Map(
        location=[centro_lat, centro_lon],
        zoom_start=12,
        tiles="CartoDB positron"   # mapa limpio y moderno
    )
    
    # ----------------------------------------------------------
    # Agregar colormap como leyenda
    # ----------------------------------------------------------
    colormap.add_to(mapa)
    
    # ----------------------------------------------------------
    # Marcadores por gasolinera
    # ----------------------------------------------------------
    for _, row in df_filtrado.iterrows():
        precio  = row[combustible]
        color   = colormap(precio)
        
        # Etiqueta de precio relativo al promedio
        if precio <= precio_avg * 0.99:
            etiqueta = "🟢 Barato"
        elif precio >= precio_avg * 1.01:
            etiqueta = "🔴 Caro"
        else:
            etiqueta = "🟡 Promedio"
        
        # Popup con info completa
        popup_html = f"""
        <div style="font-family: Arial; min-width: 200px;">
            <h4 style="margin:0; color:#2c3e50;">{row['Nombre']}</h4>
            <hr style="margin:4px 0;">
            <p style="margin:2px 0;">📍 {row['Direccion']}</p>
            <p style="margin:2px 0;">⛽ <b>Magna:</b>   ${row['Magna']:.2f}</p>
            <p style="margin:2px 0;">⭐ <b>Premium:</b> ${row['Premium']:.2f}</p>
            <p style="margin:2px 0;">🚛 <b>Diesel:</b>  
                {'$' + f"{row['Diesel']:.2f}" if not pd.isna(row['Diesel']) else 'N/A'}
            </p>
            <hr style="margin:4px 0;">
            <p style="margin:2px 0;">{etiqueta} vs promedio</p>
        </div>
        """
        
        folium.CircleMarker(
            location=[row["Latitud"], row["Longitud"]],
            radius=8,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=f"{row['Nombre']} | ${precio:.2f}"
        ).add_to(mapa)
    
    return mapa


# ============================================================
# 3. GENERAR MAPAS
# ============================================================

# --- Magna ---
mapa_magna = crear_mapa_gasolineras(df_geocoded, combustible="Magna")
mapa_magna.save("mapa_magna.html")

# --- Premium ---
mapa_premium = crear_mapa_gasolineras(df_geocoded, combustible="Premium")
mapa_premium.save("mapa_premium.html")

# --- Diesel ---
mapa_diesel = crear_mapa_gasolineras(df_geocoded, combustible="Diesel")
mapa_diesel.save("mapa_diesel.html")

print("\n✅ Mapas guardados: mapa_magna.html | mapa_premium.html | mapa_diesel.html")

# Mostrar en Jupyter
mapa_magna

⛽ Combustible : Magna
🟢 Más barato  : $22.89
🔴  Más caro    : $24.99
📊 Promedio    : $23.74
📍 Gasolineras : 155
⛽ Combustible : Premium
🟢 Más barato  : $26.53
🔴  Más caro    : $29.99
📊 Promedio    : $28.86
📍 Gasolineras : 156
⛽ Combustible : Diesel
🟢 Más barato  : $27.99
🔴  Más caro    : $29.99
📊 Promedio    : $29.26
📍 Gasolineras : 76

✅ Mapas guardados: mapa_magna.html | mapa_premium.html | mapa_diesel.html


In [47]:
df_geocoded

,Numero,Nombre,Direccion,Diesel,Magna,Premium,Direccion_completa,Latitud,Longitud
0,PL/10156/EXP/ES/2015,FEDERICO LÓPEZ LÓPEZ,Carretera Ures-Hermosillo Km 3.5,28.99,NaN,28.99,"Carretera Ures-Hermosillo Km 3.5, hermosillo, ...",29.082093,-110.942156
1,PL/10244/EXP/ES/2015,GP GASOLINERA PERINORTE S.A. DE C.V.,Avenida Camino del Seri 1242,NaN,NaN,29.00,"Avenida Camino del Seri 1242, hermosillo, sono...",29.056668,-111.015853
2,PL/10321/EXP/ES/2015,SERVICIOS DEL REAL DEL ALAMITO S.A. DE C.V.,Carretera Hermosillo Ures Entronq.Carretera de...,28.99,23.89,28.79,Carretera Hermosillo Ures Entronq.Carretera de...,29.145423,-111.023923
3,PL/10327/EXP/ES/2015,SERVICIO EL KENO S.A. DE C.V.,José Ma Mendoza 794,28.59,22.93,27.99,"José Ma Mendoza 794, hermosillo, sonora, mexico",29.101898,-110.993253
4,PL/10366/EXP/ES/2015,"ESTACION PIRU, S.A. DE C.V.",Xolotl No. 244,NaN,23.99,27.99,"Xolotl No. 244, hermosillo, sonora, mexico",29.024009,-110.946251
...,...,...,...,...,...,...,...,...,...
152,PL/8914/EXP/ES/2015,GASERVICIO EL LLANITO QUIROGA SA DE CV,Periférico Poniente Esquina Avenida Escuinapa ...,29.99,23.99,29.99,Periférico Poniente Esquina Avenida Escuinapa ...,29.148532,-110.980895
153,PL/9031/EXP/ES/2015,EDUARDO PERALTA ENCINAS,Avenida Periférico Norte No. 371,NaN,23.49,29.00,"Avenida Periférico Norte No. 371, hermosillo, ...",29.112167,-110.974393
154,PL/9577/EXP/ES/2015,ESTACIÓN PIRU S.A. DE C.V.,De La Reforma Esquina Con Aguascalientes No. 79,NaN,23.99,27.99,De La Reforma Esquina Con Aguascalientes No. 7...,29.095682,-110.959122
155,PL/9672/EXP/ES/2015,SERVICIOS DEL RÍO RG S.A. DE C.V.,Boulevard Paseo del Canal Sur Al Este Fraccion...,NaN,24.29,29.99,Boulevard Paseo del Canal Sur Al Este Fraccion...,29.064734,-110.996116


## Pipeline

In [1]:
import os
import time
import json
import requests
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sina.config.paths import GAS_DATA
from sina.config.credentials import gasolina_api_rest, google_api_key

MUNICIPIOS_JSON = GAS_DATA / Path("catalogo_municipios.json")

base_dir = GAS_DATA
CIUDAD  = "hermosillo"
ESTADO  = "sonora"
PAIS    = "mexico"

CACHE_FILE = GAS_DATA / CIUDAD / Path(f"gasolineras_{CIUDAD}.json")

In [2]:
with open(CACHE_FILE, "r", encoding="utf-8") as f:
    gas_dict = json.load(f)

In [3]:
with open(MUNICIPIOS_JSON, "r", encoding="utf-8") as f:
    mun_dict = json.load(f)

In [4]:
mun_dict[ESTADO]['id']

'26'

In [5]:
mun_dict[ESTADO]['municipios'][CIUDAD].get('id')

'030'

In [6]:
def gas_prices(
        estado: str,
        ciudad: str
) -> dict:
    params = {
        "entidadId"     : mun_dict[estado]['id'],
        "municipioId"   : mun_dict[estado]['municipios'][ciudad].get('id')
    }

    headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer"   : "https://www.cne.gob.mx/ConsultaPrecios/GasolinasyDiesel/GasolinasyDiesel.html"
    }
    response = requests.get(gasolina_api_rest, params=params, headers=headers)
    print(f"Status  : {response.status_code}")
    print(f"Size    : {len(response.content) / 1024:.1f} KB")  
    print(f"Content : {response.headers.get('Content-Type')}")

    data = response.json()
    print(f"\nTipo    : {type(data)}")

    if isinstance(data, list):
        print(f"Registros : {len(data)}")
        print(f"\nPrimer registro:")
        print(json.dumps(data[0], indent=2, ensure_ascii=False))

    elif isinstance(data, dict):
        print(f"Keys : {list(data.keys())}")
    
    return dict(data)

In [7]:
data = gas_prices(ESTADO, CIUDAD)
data['Value']

Status  : 200
Size    : 111.7 KB
Content : application/json; charset=utf-8

Tipo    : <class 'dict'>
Keys : ['Success', 'Errors', 'Value']


[{'Numero': 'PL/3438/EXP/ES/2015',
  'Direccion': 'Ángel García Aburto Y Dr. Domingo Olivares',
  'Producto': 'Gasolinas',
  'SubProducto': 'Premium (con un índice de octano ([RON+MON]/2) mínimo de 91)',
  'PrecioVigente': 29.99,
  'EntidadFederativaId': 26,
  'MunicipioId': 30,
  'Nombre': 'SERVICIOS Z3 S.A. DE C.V.'},
 {'Numero': 'PL/3438/EXP/ES/2015',
  'Direccion': 'Ángel García Aburto Y Dr. Domingo Olivares',
  'Producto': 'Gasolinas',
  'SubProducto': 'Regular (con un índice de octano ([RON+MON]/2) mínimo de 87)',
  'PrecioVigente': 23.99,
  'EntidadFederativaId': 26,
  'MunicipioId': 30,
  'Nombre': 'SERVICIOS Z3 S.A. DE C.V.'},
 {'Numero': 'PL/20952/EXP/ES/2018',
  'Direccion': 'AVE MAR DE CORTES NÚMERO 8910',
  'Producto': 'Diésel',
  'SubProducto': 'Diésel Automotriz [contenido mayor de azufre a 15 mg/kg y contenido máximo de azufre de 500 mg/kg]',
  'PrecioVigente': 29.9,
  'EntidadFederativaId': 26,
  'MunicipioId': 30,
  'Nombre': 'ESTACION DE SERVICIO ERSAL SA DE CV'},
 {

In [8]:
df_cne = pd.DataFrame(data["Value"])
print(df_cne.shape)
print(df_cne.columns.tolist())
df_cne.head()

(391, 8)
['Numero', 'Direccion', 'Producto', 'SubProducto', 'PrecioVigente', 'EntidadFederativaId', 'MunicipioId', 'Nombre']


,Numero,Direccion,Producto,SubProducto,PrecioVigente,EntidadFederativaId,MunicipioId,Nombre
0,PL/3438/EXP/ES/2015,Ángel García Aburto Y Dr. Domingo Olivares,Gasolinas,Premium (con un índice de octano ([RON+MON]/2)...,29.99,26,30,SERVICIOS Z3 S.A. DE C.V.
1,PL/3438/EXP/ES/2015,Ángel García Aburto Y Dr. Domingo Olivares,Gasolinas,Regular (con un índice de octano ([RON+MON]/2)...,23.99,26,30,SERVICIOS Z3 S.A. DE C.V.
2,PL/20952/EXP/ES/2018,AVE MAR DE CORTES NÚMERO 8910,Diésel,Diésel Automotriz [contenido mayor de azufre a...,29.90,26,30,ESTACION DE SERVICIO ERSAL SA DE CV
3,PL/20952/EXP/ES/2018,AVE MAR DE CORTES NÚMERO 8910,Gasolinas,Premium (con un índice de octano ([RON+MON]/2)...,29.10,26,30,ESTACION DE SERVICIO ERSAL SA DE CV
4,PL/20952/EXP/ES/2018,AVE MAR DE CORTES NÚMERO 8910,Gasolinas,Regular (con un índice de octano ([RON+MON]/2)...,23.99,26,30,ESTACION DE SERVICIO ERSAL SA DE CV


In [11]:
mapa_productos = {
    sp: ("Premium" if "Premium" in sp 
         else "Magna" if "Regular" in sp 
         else "Diesel" if "Diésel" in sp  
         else "Otro")
    for sp in df_cne["SubProducto"].unique()
}

print("Mapeo encontrado:")
for k, v in mapa_productos.items():
    print(f"  {v:10} ← {k}")

df_cne["Combustible"] = df_cne["SubProducto"].map(mapa_productos)

df_pivot = df_cne.pivot_table(
    index   = ["Numero", "Nombre", "Direccion"],
    columns = "Combustible",
    values  = "PrecioVigente",
    aggfunc = "first"
).reset_index()

df_pivot.columns.name = None

for col in ["Magna", "Premium", "Diesel"]:
    if col not in df_pivot.columns:
        df_pivot[col] = None

df_pivot.head(10)

Mapeo encontrado:
  Premium    ← Premium (con un índice de octano ([RON+MON]/2) mínimo de 91)
  Magna      ← Regular (con un índice de octano ([RON+MON]/2) mínimo de 87)
  Diesel     ← Diésel Automotriz [contenido mayor de azufre a 15 mg/kg y contenido máximo de azufre de 500 mg/kg]
  Diesel     ← Diésel
  Premium    ← Premium (con contenido mínimo de 92 octanos)
  Magna      ← Regular (con contenido menor a 92 octanos)
  Diesel     ← Diésel de Ultra Bajo Azufre (DUBA) [contenido máximo de azufre de 15 mg/kg]


,Numero,Nombre,Direccion,Diesel,Magna,Premium
0,PL/10156/EXP/ES/2015,FEDERICO LÓPEZ LÓPEZ,Carretera Ures-Hermosillo Km 3.5,28.99,NaN,28.99
1,PL/10244/EXP/ES/2015,GP GASOLINERA PERINORTE S.A. DE C.V.,Avenida Camino del Seri 1242,NaN,NaN,29.00
2,PL/10321/EXP/ES/2015,SERVICIOS DEL REAL DEL ALAMITO S.A. DE C.V.,Carretera Hermosillo Ures Entronq.Carretera de...,28.99,23.89,28.79
3,PL/10327/EXP/ES/2015,SERVICIO EL KENO S.A. DE C.V.,José Ma Mendoza 794,28.59,22.93,27.99
4,PL/10366/EXP/ES/2015,"ESTACION PIRU, S.A. DE C.V.",Xolotl No. 244,NaN,23.99,27.99
5,PL/10370/EXP/ES/2015,MIGRAG GRUPO CONSTUCTOR S.A. DE C.V.,Periférico Norte 4,NaN,22.99,27.90
6,PL/10696/EXP/ES/2015,GASOLINERA RS S.A. DE C.V.,Margarita Maza de Juárez Y José López Portillo Sn,NaN,23.49,27.99
7,PL/10740/EXP/ES/2015,ESTACIÓN DE SERVICIO QUIROGA S.A. DE C.V.,Boulevard Luis Donaldo Colosio 862 Esquinaboul...,NaN,22.99,28.99
8,PL/10761/EXP/ES/2015,"CORPORATIVO ENERVISION, S.A.P.I. DE C.V.",Boulevard López Portillo No. 189,29.99,24.29,29.99
9,PL/10763/EXP/ES/2015,"CORPORATIVO ENERVISION, S.A.P.I. DE C.V.",Boulevard Gomez Farias Y Boulevard Ignacio Sot...,NaN,24.29,29.99


In [12]:
gas_dict['PL/20952/EXP/ES/2018']

{'Nombre': 'ESTACION DE SERVICIO ERSAL SA DE CV',
 'Direccion': 'AVE MAR DE CORTES NÚMERO 8910',
 'Direccion_completa': 'AVE MAR DE CORTES NÚMERO 8910, hermosillo, sonora, mexico',
 'Latitud': 29.084843,
 'Longitud': -110.9880654}

In [18]:
gas_dict[df_pivot['Numero'][0]]['Latitud']

29.0820931

In [20]:
df_pivot["Latitud"] = df_pivot["Numero"].map(
    lambda x: gas_dict.get(x, {}).get("Latitud")
)

df_pivot["Longitud"] = df_pivot["Numero"].map(
    lambda x: gas_dict.get(x, {}).get("Longitud")
)

In [23]:
print(f"Con coordenadas    : {df_pivot[['Latitud', 'Longitud']].notna().all(axis=1).sum()}")
print(f"Sin coordenadas    : {df_pivot[['Latitud', 'Longitud']].isna().any(axis=1).sum()}")
df_pivot.head(3)

Con coordenadas    : 157
Sin coordenadas    : 0


,Numero,Nombre,Direccion,Diesel,Magna,Premium,Latitud,Longitud
0,PL/10156/EXP/ES/2015,FEDERICO LÓPEZ LÓPEZ,Carretera Ures-Hermosillo Km 3.5,28.99,NaN,28.99,29.082093,-110.942156
1,PL/10244/EXP/ES/2015,GP GASOLINERA PERINORTE S.A. DE C.V.,Avenida Camino del Seri 1242,NaN,NaN,29.00,29.056668,-111.015853
2,PL/10321/EXP/ES/2015,SERVICIOS DEL REAL DEL ALAMITO S.A. DE C.V.,Carretera Hermosillo Ures Entronq.Carretera de...,28.99,23.89,28.79,29.145423,-111.023923


In [24]:
output_path  = GAS_DATA / CIUDAD / f"gas_prices_{CIUDAD}.csv"
df_pivot.to_csv(output_path, index=False, encoding="utf-8-sig")  

In [ ]:
df_pivot

,Numero,Nombre,Direccion,Diesel,Magna,Premium,Latitud,Longitud
0,PL/10156/EXP/ES/2015,FEDERICO LÓPEZ LÓPEZ,Carretera Ures-Hermosillo Km 3.5,28.99,NaN,28.99,29.082093,-110.942156
